# Notebook 01 — Setup, Data Audit & EDA
**Project:** Unlocking Behavioral Intelligence in Airline Loyalty Programs  
**Club:** Consulting & Analytics Club, IIT Guwahati  
**Notebook purpose:** Load all four data files, audit data quality, document cleaning decisions, and produce exploratory visuals that motivate the churn definition and segmentation approach in later notebooks.


## 1. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Paths ── update DATA_DIR if your folder layout is different
DATA_DIR = Path("data/raw")          # put the four CSV files here
FIG_DIR  = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})

print("Libraries loaded.")


## 2. Load data

In [ ]:
lh  = pd.read_csv(DATA_DIR / "Customer_Loyalty_History.csv")
fa  = pd.read_csv(DATA_DIR / "Customer_Flight_Activity.csv")
cal = pd.read_csv(DATA_DIR / "Calendar.csv")
dd  = pd.read_csv(DATA_DIR / "Airline_Loyalty_Data_Dictionary.csv")

print(f"Loyalty History  : {lh.shape[0]:,} rows × {lh.shape[1]} cols")
print(f"Flight Activity  : {fa.shape[0]:,} rows × {fa.shape[1]} cols")
print(f"Calendar         : {cal.shape[0]:,} rows × {cal.shape[1]} cols")
dd


## 3. Schema & null audit

In [ ]:
print("=" * 50)
print("LOYALTY HISTORY — dtypes & nulls")
print("=" * 50)
audit_lh = pd.DataFrame({
    "dtype"    : lh.dtypes,
    "non_null" : lh.notna().sum(),
    "null_pct" : (lh.isna().mean() * 100).round(1),
})
display(audit_lh)

print()
print("=" * 50)
print("FLIGHT ACTIVITY — dtypes & nulls")
print("=" * 50)
audit_fa = pd.DataFrame({
    "dtype"    : fa.dtypes,
    "non_null" : fa.notna().sum(),
    "null_pct" : (fa.isna().mean() * 100).round(1),
})
display(audit_fa)


**Data quality notes (document your decisions here):**

| Issue | Decision |
|---|---|
| `Salary` — 25.3 % null | Retain column; impute with median-by-card-tier for modelling. Do NOT drop rows. |
| `Cancellation Year/Month` — 87.7 % null | Null = still active. This is the formal churn label (Definition A). |
| Flight activity covers **2017–2018 only** | All time-series features are constructed within this 24-month window. |
| Zero-flight months (54.5 % of activity rows) | Retain — absence of activity is a signal, not missing data. |


## 4. Key structural facts

In [ ]:
# 4a. Member overlap across tables
in_lh = set(lh["Loyalty Number"])
in_fa = set(fa["Loyalty Number"])
assert in_lh == in_fa, "Member sets differ — investigate!"
print(f"✓ All {len(in_lh):,} members appear in both tables.")

# 4b. Time span
print(f"\nFlight activity window : {fa['Year'].min()}-{fa['Month'].min():02d}  →  {fa['Year'].max()}-{fa['Month'].max():02d}")
print(f"Enrollment years       : {lh['Enrollment Year'].min()} – {lh['Enrollment Year'].max()}")
print(f"Cancellation years     : {lh['Cancellation Year'].dropna().min():.0f} – {lh['Cancellation Year'].dropna().max():.0f}")

# 4c. Cancelled vs active
n_cancelled = lh["Cancellation Year"].notna().sum()
n_active    = lh["Cancellation Year"].isna().sum()
print(f"\nActive members         : {n_active:,}  ({n_active/len(lh)*100:.1f} %)")
print(f"Cancelled members      : {n_cancelled:,}  ({n_cancelled/len(lh)*100:.1f} %)")


## 5. Cleaning steps

In [ ]:
# 5a. Loyalty History — salary imputation
lh["Salary_imputed"] = lh.groupby("Loyalty Card")["Salary"].transform(
    lambda x: x.fillna(x.median())
)

# 5b. Flight Activity — add a date column for time-series work
fa["Date"] = pd.to_datetime(
    fa["Year"].astype(str) + "-" + fa["Month"].astype(str).str.zfill(2) + "-01"
)

# 5c. Flag if member is churned (formal definition A)
lh["churned_formal"] = lh["Cancellation Year"].notna().astype(int)

# 5d. Compute member-level tenure (months from enrollment to last activity / cancellation)
# Enrollment as date (assume day=1)
lh["enrollment_date"] = pd.to_datetime(
    lh["Enrollment Year"].astype(str) + "-" + lh["Enrollment Month"].astype(str).str.zfill(2) + "-01"
)

print("Cleaning done. New columns added:")
print("  lh : Salary_imputed, churned_formal, enrollment_date")
print("  fa : Date")


## 6. Exploratory analysis

### 6a. Loyalty card & CLV distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Card tier counts
card_order = ["Star", "Nova", "Aurora"]
card_counts = lh["Loyalty Card"].value_counts().reindex(card_order)
axes[0].bar(card_order, card_counts.values, color=["#5DCAA5","#7F77DD","#EF9F27"], edgecolor="none")
axes[0].set_title("Members by loyalty card tier")
axes[0].set_ylabel("Count")
for i, v in enumerate(card_counts.values):
    axes[0].text(i, v + 50, f"{v:,}", ha="center", fontsize=10)

# CLV by card tier
lh.boxplot(column="CLV", by="Loyalty Card", ax=axes[1], order=card_order,
           boxprops=dict(color="#5F5E5A"),
           medianprops=dict(color="#D85A30", linewidth=2))
axes[1].set_title("CLV distribution by card tier")
axes[1].set_xlabel("")
axes[1].set_ylabel("CLV (CAD)")
plt.suptitle("")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_card_clv.png", bbox_inches="tight")
plt.show()
print("Saved → reports/figures/01_card_clv.png")


### 6b. Monthly flight activity trend (2017–2018)

In [ ]:
monthly = (
    fa.groupby(["Year","Month"])["Total Flights"]
    .sum()
    .reset_index()
)
monthly["label"] = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(monthly["label"], monthly["Total Flights"], marker="o", markersize=4,
        color="#534AB7", linewidth=1.5)
ax.set_title("Total flights booked per month (all members, 2017–2018)")
ax.set_ylabel("Total flights")
ax.set_xlabel("")
plt.xticks(rotation=45, ha="right", fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(FIG_DIR / "02_monthly_flights.png", bbox_inches="tight")
plt.show()


### 6c. Churn rate by card tier & enrollment type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# By card tier
churn_card = lh.groupby("Loyalty Card")["churned_formal"].mean().reindex(card_order) * 100
axes[0].bar(card_order, churn_card.values, color=["#5DCAA5","#7F77DD","#EF9F27"], edgecolor="none")
axes[0].set_title("Formal churn rate by card tier")
axes[0].set_ylabel("Churn rate (%)")
for i, v in enumerate(churn_card.values):
    axes[0].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=10)

# By enrollment type
churn_enr = lh.groupby("Enrollment Type")["churned_formal"].mean() * 100
axes[1].bar(churn_enr.index, churn_enr.values, color=["#1D9E75","#EF9F27"], edgecolor="none")
axes[1].set_title("Formal churn rate by enrollment type")
axes[1].set_ylabel("Churn rate (%)")
for i, v in enumerate(churn_enr.values):
    axes[1].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIG_DIR / "03_churn_by_segment.png", bbox_inches="tight")
plt.show()


### 6d. Behavioral activity — active months distribution

In [ ]:
member_summary = (
    fa.groupby("Loyalty Number")
    .agg(
        months_with_flights=("Total Flights", lambda x: (x > 0).sum()),
        total_flights=("Total Flights", "sum"),
        total_distance=("Distance", "sum"),
        total_pts_acc=("Points Accumulated", "sum"),
        total_pts_red=("Points Redeemed", "sum"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(member_summary["months_with_flights"], bins=25, color="#378ADD", edgecolor="white")
axes[0].set_title("Active months per member (out of 24)")
axes[0].set_xlabel("Months with ≥1 flight")
axes[0].set_ylabel("Member count")

axes[1].hist(member_summary["total_flights"].clip(upper=80), bins=30, color="#D85A30", edgecolor="white")
axes[1].set_title("Total flights per member (2017–2018, capped at 80)")
axes[1].set_xlabel("Total flights")
axes[1].set_ylabel("Member count")

plt.tight_layout()
plt.savefig(FIG_DIR / "04_activity_dist.png", bbox_inches="tight")
plt.show()

print("Members with 0 active months:", (member_summary["months_with_flights"] == 0).sum())
print("Members with >18 active months:", (member_summary["months_with_flights"] > 18).sum())


## 7. Churn definitions — two competing versions

**Core constraint:** The flight activity window is only 2017–2018.  
Any churn definition must be constructed from variables available at a simulated prediction point — we will treat **end of 2017 (Dec 2017)** as the prediction cutoff and ask: *will this member disengage during 2018?*

| Definition | Logic | Label as churned if… |
|---|---|---|
| **A — Formal** | Used official cancellation record | `Cancellation Year` is not null |
| **B — Behavioral** | Constructed from flight activity | Member had ≥1 flight/month in H1 2017 but zero flights in H2 2017 + all of 2018 |

Definition B is more operationally useful: it catches members who quietly go silent before formally cancelling.  
We will build both, test their overlap, and argue for B in the technical report.


In [ ]:
# ── Definition B construction ──────────────────────────────────────────────
# "Active in H1 2017" = at least 1 month with flights in Jan–Jun 2017
# "Silent in observation window" = zero flights Jul 2017 – Dec 2018 (18 months)

h1_2017 = fa[(fa["Year"] == 2017) & (fa["Month"] <= 6)]
obs_window = fa[~((fa["Year"] == 2017) & (fa["Month"] <= 6))]  # Jul 2017 onwards

active_h1 = h1_2017.groupby("Loyalty Number")["Total Flights"].sum().reset_index()
active_h1.columns = ["Loyalty Number", "flights_h1_2017"]
active_h1["active_in_h1"] = active_h1["flights_h1_2017"] > 0

silent_obs = obs_window.groupby("Loyalty Number")["Total Flights"].sum().reset_index()
silent_obs.columns = ["Loyalty Number", "flights_obs_window"]
silent_obs["silent_obs"] = silent_obs["flights_obs_window"] == 0

churn_b = active_h1.merge(silent_obs, on="Loyalty Number")
churn_b["churned_behavioral"] = (churn_b["active_in_h1"] & churn_b["silent_obs"]).astype(int)

print("Definition B — behavioral churn rate:",
      f"{churn_b['churned_behavioral'].mean()*100:.1f}%",
      f"({churn_b['churned_behavioral'].sum():,} members)")

# Merge both labels onto lh
lh = lh.merge(churn_b[["Loyalty Number","churned_behavioral"]], on="Loyalty Number", how="left")
lh["churned_behavioral"] = lh["churned_behavioral"].fillna(0).astype(int)

# Overlap analysis
from_formal   = lh["churned_formal"].sum()
from_behav    = lh["churned_behavioral"].sum()
both          = (lh["churned_formal"] & lh["churned_behavioral"]).sum()
print(f"\nDefinition A (formal)    : {from_formal:,} churned")
print(f"Definition B (behavioral): {from_behav:,} churned")
print(f"Overlap (both)           : {both:,}")
print(f"Caught by B only         : {from_behav - both:,}  ← members who went silent but didn't cancel")


### Argument for Definition B

Definition A (formal cancellation) only captures members who took the explicit step of cancelling — a fraction of true disengagement. Definition B captures the more common pattern: members who simply stop flying without formally cancelling. This is behaviourally and commercially the more dangerous group, as they represent revenue lost without any early warning signal.

**We proceed with Definition B as the primary churn label.** Definition A will be used as a validation check — a good model should have high recall on formally-cancelled members even when trained on behavioral labels.


## 8. Save cleaned data

In [ ]:
out = Path("data/processed")
out.mkdir(parents=True, exist_ok=True)

lh.to_csv(out / "loyalty_history_clean.csv", index=False)
fa.to_csv(out / "flight_activity_clean.csv", index=False)
member_summary.to_csv(out / "member_activity_summary.csv", index=False)

print("Saved:")
print("  data/processed/loyalty_history_clean.csv")
print("  data/processed/flight_activity_clean.csv")
print("  data/processed/member_activity_summary.csv")
print("\nNotebook 01 complete. Proceed to 02_feature_engineering.ipynb")
